# Reproducible StreamFlix analysis

This is the canonical, database-free analysis path. It reads the checked-in CSV snapshots, validates the core relationships, reproduces the headline descriptive metrics, and calculates the genre-level subscriber affinity comparison used by the final Power BI report.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_repository_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')

ROOT = find_repository_root()
ROOT

In [ ]:
users = pd.read_csv(ROOT / 'data/raw/Users.csv')
movies = pd.read_csv(ROOT / 'data/processed/movies_clean.csv')
ratings = pd.read_csv(ROOT / 'data/processed/ratings_clean.csv', parse_dates=['Timestamp'], dayfirst=True)
genres = pd.read_csv(ROOT / 'data/processed/genres.csv')
movie_genres = pd.read_csv(ROOT / 'data/processed/movie_genres.csv')

assert users['UserID'].is_unique
assert movies['MovieID'].is_unique
assert ratings['RatingID'].is_unique
assert genres['GenreID'].is_unique
assert not ratings['UserID'].isin(users['UserID']).eq(False).any()
assert not ratings['MovieID'].isin(movies['MovieID']).eq(False).any()
assert not movie_genres['MovieID'].isin(movies['MovieID']).eq(False).any()
assert not movie_genres['GenreID'].isin(genres['GenreID']).eq(False).any()

{
    'users': len(users),
    'movies': len(movies),
    'ratings': len(ratings),
    'genres': len(genres),
    'movie_genres': len(movie_genres),
}

In [ ]:
rated_users = ratings['UserID'].nunique()
rated_movies = ratings['MovieID'].nunique()
headline_metrics = pd.Series({
    'Registered users': len(users),
    'Users who rated': rated_users,
    'Movies': len(movies),
    'Movies rated (%)': 100 * rated_movies / len(movies),
    'Clean ratings': len(ratings),
    'Ratings per registered user': len(ratings) / len(users),
    'Average rating': ratings['Rating'].mean(),
})
headline_metrics.round(2)

In [ ]:
age_bins = [-float('inf'), 10, 20, 30, 40, 50, 60, float('inf')]
age_labels = ['0-10', '11-20', '21-30', '31-40', '41-50', '51-60', '61+']
audience = users.assign(
    AgeBand=pd.cut(users['Age'], bins=age_bins, labels=age_labels)
)

display(audience['AgeBand'].value_counts(sort=False).rename('Users').to_frame())
display(users['SubscriptionStatus'].value_counts().rename('Users').to_frame())
display(users['Device'].value_counts().rename('Users').to_frame())

In [ ]:
rating_genres = (
    ratings
    .merge(users[['UserID', 'SubscriptionStatus']], on='UserID', validate='many_to_one')
    .merge(movie_genres, on='MovieID', validate='many_to_many')
    .merge(genres, on='GenreID', validate='many_to_one')
)

genre_volume = (
    rating_genres.groupby('Genre_Name', as_index=False)
    .agg(RatingCount=('RatingID', 'count'), AverageRating=('Rating', 'mean'))
    .sort_values('RatingCount', ascending=False)
)
genre_volume

## Subscriber affinity, not conversion

The dashboard scatterplot groups by **genre**, not title. `SubscriberAffinityGap` is the difference between the subscriber and free-user average ratings observed in the current cohorts. It is descriptive: the data has no subscription-event time, historical status, content-exposure sequence, or control group, so the gap cannot establish that a genre caused conversion.

In [ ]:
cohort_ratings = (
    rating_genres.groupby(['Genre_Name', 'SubscriptionStatus'])['Rating']
    .agg(['mean', 'count'])
    .reset_index()
)
means = cohort_ratings.pivot(index='Genre_Name', columns='SubscriptionStatus', values='mean')
counts = cohort_ratings.pivot(index='Genre_Name', columns='SubscriptionStatus', values='count')
genre_views = (
    movie_genres.merge(movies[['MovieID', 'Total Views']], on='MovieID', validate='many_to_one')
    .merge(genres, on='GenreID', validate='many_to_one')
    .groupby('Genre_Name')['Total Views'].sum()
)

affinity = pd.DataFrame({
    'FreeAverageRating': means['Free'],
    'SubscriberAverageRating': means['Subscriber'],
    'FreeRatingCount': counts['Free'].astype(int),
    'SubscriberRatingCount': counts['Subscriber'].astype(int),
    'CatalogTotalViews': genre_views,
})
affinity['SubscriberAffinityGap'] = (
    affinity['SubscriberAverageRating'] - affinity['FreeAverageRating']
)
affinity.sort_values('SubscriberAffinityGap', ascending=False).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sizes = 50 + 950 * affinity['CatalogTotalViews'] / affinity['CatalogTotalViews'].max()
ax.scatter(
    affinity['FreeAverageRating'],
    affinity['SubscriberAverageRating'],
    s=sizes,
    alpha=0.7,
)
for genre, row in affinity.iterrows():
    ax.annotate(genre, (row['FreeAverageRating'], row['SubscriberAverageRating']), fontsize=8)
limits = [
    min(affinity['FreeAverageRating'].min(), affinity['SubscriberAverageRating'].min()) - 0.03,
    max(affinity['FreeAverageRating'].max(), affinity['SubscriberAverageRating'].max()) + 0.03,
]
ax.plot(limits, limits, linestyle='--', color='grey', linewidth=1)
ax.set(
    xlim=limits,
    ylim=limits,
    xlabel='Free-user average rating',
    ylabel='Subscriber average rating',
    title='Observed genre affinity by current subscription cohort',
)
plt.tight_layout()
plt.show()

## Interpretation boundary

Use subscriber-skewed genres as hypotheses for content research, not as proof of conversion. Testing a conversion effect requires subscription-event timestamps, pre/post subscription state, title-level exposure or watch events, temporal ordering, and preferably a controlled experiment or a justified observational design.